<a href="https://colab.research.google.com/github/khiemp110/template/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## KoboldCpp + StableUI trên Colab

1. Chọn **GPU runtime**. Bật `UsePreset`, chọn `Template`, rồi chạy cell bên dưới. Tắt `UsePreset` để dùng các checkbox thủ công.
2. `AllowSaveToGoogleDrive = True` mặc định: cấp quyền Drive lần đầu. Model lưu trong **MyDrive/Colab Notebooks/KoboldCPP/models/**. Phiên sau kiểm tra SHA256 rồi dùng lại; tải dở được tiếp tục. Mọi tải và đọc model chạy bằng mạng của **máy Colab**.
3. Mở link tunnel do KoboldCpp cung cấp, thêm **`/sdui/`** để vào **StableUI** tạo ảnh.
4. Có Secret `CLOUDFLARE_API_KEY`: dùng named tunnel đã route tới `http://localhost:5001`. Nếu không, KoboldCpp dùng `--remotetunnel`. Bấm **Stop** để dừng phiên và tunnel do cell tạo.
5. `DeleteExistingModels` chỉ dọn cache tạm `/content/koboldcpp-cache`. Muốn xóa cache Drive, tự quản lý thư mục trên Drive. Tắt lưu Drive thì cache mất khi runtime bị xóa.

Đọc model nhiều GB từ Drive và kiểm tra SHA256 có thể mất thời gian, nhưng không tải lại trọng số từ Hugging Face.

**Preset mặc định: MiniMax H3 Image (Q4 L4).** Model ảnh dùng AtomicChat AD-Q4_K (~11,57 GB), text encoder H3 GGUF Q4_K_M (~18,22 GB) và Image VAE FP16 (~1,17 GB). Tổng cache khoảng **31 GB**. Preset dành cho **L4 24 GB**; notebook không chặn chạy theo dung lượng RAM/VRAM. Text encoder và VAE chạy CPU; DiT dùng CUDA, CPU offload, Flash Attention và VAE tiling để giảm VRAM.

**Vẫn dùng KoboldCpp + StableUI:** preset này build bản native từ nhánh cập nhật SD của maintainer (PR #2576), ghép patch H3 Image upstream ở commit cố định. Lần đầu build có thể mất nhiều phút; mã nguồn khoảng 70 MB và patch khoảng 1 MB được tải bằng mạng Colab. Build hoàn tất được lưu tại `Colab Notebooks/KoboldCPP/runtime/` để phiên sau dùng lại nếu CUDA/GPU tương thích. Không cài backend Python tạo ảnh khác.

StableUI mở với **1024×1024, 6 bước, Euler, CFG 1** cho preset này. Đây là cấu hình khởi đầu cho L4; chưa benchmark tạo ảnh trên L4 thực tế trong phiên sửa notebook này. Các preset khác tiếp tục dùng binary KoboldCpp phát hành chính thức.

Nguồn model: [AtomicChat GGUF](https://huggingface.co/AtomicChat/MiniMax-H3-Image-GGUF), [text encoder](https://huggingface.co/leejet/MiniMax-H3-GGUF). Nguồn backend: [KoboldCpp SD update](https://github.com/LostRuins/koboldcpp/pull/2576), [H3 Image upstream](https://github.com/leejet/stable-diffusion.cpp/commit/32d516c0197ed5ca31305b8575e14eb1451fad33).

[Tài liệu KoboldCpp / StableUI](https://koboldcpp.net/docs/features/image-generation/).


In [ ]:
#@title Âm thanh tùy chọn (không đảm bảo giữ runtime) { display-mode: "form" }
%%html
<audio src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" loop controls></audio>

In [ ]:
# @title <b>v-- Chọn Model bên dưới rồi bấm Play để chạy KoboldCpp nhé peko!</b> { display-mode: "form" }
# @markdown Nhớ bấm nút Play khởi động lại mỗi khi ngươi đổi cấu hình đó!

import hashlib
import json
import os
from pathlib import Path
import shlex
import signal
import shutil
import subprocess
from urllib.parse import unquote, urlparse
from google.colab import drive, userdata

# @markdown ### 1. Chọn nhanh Preset có sẵn (Tùy chọn)
UsePreset = True # @param {type:"boolean"}
Template = "MiniMax H3 Image (Q4 L4)" # @param ["Gemma4 E4B Uncensored (General)", "Tiefighter 13B (General)", "Estopia 13B (General)", "Qwen3 VL 8B (Vision)", "Cydonia 24B (Roleplay)", "Broken Tutu 24B (Roleplay)", "Dans PersonalityEngine 24B (Roleplay)", "Harbinger 24B (Instruct)", "Muse 12B (Instruct)", "Mistral Small 24B (Corporate)", "GPT OSS 20B (Corporate)", "Gemma3 12B (Corporate)", "GLM4 9b chat (Corporate)", "Z-Image Turbo (Image)", "Flux Klein 4B (Image)", "Qwen Image 2.1 (Image)", "Qwen Image 2.1 Max (Q8 L4)", "MiniMax H3 Image (Q4 L4)"]

# @markdown ### 2. Cấu hình thủ công (chỉ áp dụng khi tắt UsePreset)
# @markdown <hr>
LoadTextModel = False # @param {type:"boolean"}
Model = "Gemma 4 12B (Q4_K_M)" # @param ["Gemma 4 12B (Q4_K_M)", "Gemma 4 E4B Uncensored (Q5_K_M)", "Tiefighter 13B (Q4_K_S)", "Estopia 13B (Q4_K_S)", "Llama 3.1 8B BookAdventures (Q6_K)", "Cydonia 24B v4.2.0 (Q4_K_S)", "Broken Tutu 24B (Q4_K_S)", "Dans PersonalityEngine 24B (Q4_K_S)", "Harbinger 24B (Q4_K_S)", "Muse 12B (Q4_K_S)", "Qwen3 VL 8B Instruct (Q6_K)", "Mistral Small 3.2 24B (Q4_K_S)", "GPT OSS 20B (MXFP4)", "Gemma 3 12B (Q4_K_S)", "Gemma 3n E4B (Q6_K)", "GLM 4 9B (Q6_K)"] {allow-input: true}
Layers = "Auto" # @param ["Auto","999"] {allow-input: true}
ContextSize = "40960" # @param ["4096","8192","12288","16384","24576","32768","40960"] {allow-input: true}

# @markdown <hr>
LoadVisionMMProjector = False # @param {type:"boolean"}
Mmproj = "Gemma 4 E4B mmproj (BF16)" # @param ["Gemma 4 E4B mmproj (BF16)", "Qwen3 VL 8B mmproj (BF16)", "LLaMA 13B mmproj v1.5 (Q4_1)", "LLaMA 3 8B mmproj (Q4_1)"] {allow-input: true}

# @markdown <hr>
LoadImgModel = True # @param {type:"boolean"}
ImgModel = "Qwen Image 2.1 (Q8_0 - Max Quality)" # @param ["Qwen Image 2.1 (Q8_0 - Max Quality)", "Qwen Image 2.1 (Q5_K_M)", "PicX Real (Q8_0)", "DreamShaper XL v2 (Q4_K)", "Flux Klein 4B (Q4_0)", "Flux Klein 9B (Q5_K_M)"] {allow-input: true}
ImgClip1 = "Qwen3 VL 8B Instruct (Q8_0 - Max Text)" # @param ["Qwen3 VL 8B Instruct (Q8_0 - Max Text)", "Qwen3 VL 8B Instruct (UD-Q4_K_XL - Standard)", "Qwen3 4B (Q4_K_M - Low VRAM)"] {allow-input: true}
ImgVAE = "Qwen Image 2.1 VAE (Texture-Fix BF16)" # @param ["Qwen Image 2.1 VAE (Texture-Fix BF16)", "Flux2 VAE"] {allow-input: true}

# @markdown <hr>
LoadSpeechModel = False # @param {type:"boolean"}
SpeechModel = "Whisper Base EN (Q5_1)" # @param ["Whisper Base EN (Q5_1)"] {allow-input: true}

# @markdown <hr>
LoadTTSModel = False # @param {type:"boolean"}
TTSModel = "Kokoro (Q4)" # @param ["Kokoro (Q4)", "OuteTTS 0.2 500M (Q4_0)"] {allow-input: true}
WavTokModel = "None" # @param ["None", "WavTokenizer Large 75 (Q4_0)"] {allow-input: true}

# @markdown <hr>
LoadEmbeddingsModel = False # @param {type:"boolean"}
EmbeddingsModel = "Snowflake Arctic Embed S (Q4_0)" # @param ["Snowflake Arctic Embed S (Q4_0)"] {allow-input: true}

# @markdown <hr>
LoadMusicModel = False # @param {type:"boolean"}
MusicArgs = "--musiclowvram --musicllm https://huggingface.co/koboldcpp/music/resolve/main/acestep-5Hz-lm-1.7B-BF16.gguf --musicembeddings https://huggingface.co/koboldcpp/music/resolve/main/Ace-Qwen3-Embedding-0.6B-BF16.gguf --musicdiffusion https://huggingface.co/koboldcpp/music/resolve/main/acestep-v15-sftturbo50-Q8_0.gguf --musicvae https://huggingface.co/koboldcpp/music/resolve/main/ace-vae-BF16.gguf"

# @markdown <hr>
# @markdown ### 3. Tiện ích hệ thống & Tunnel
# @markdown DeleteExistingModels chỉ xóa cache tạm /content/koboldcpp-cache, không xóa model trên Drive.
DeleteExistingModels = False # @param {type:"boolean"}
Multiplayer = False # @param {type:"boolean"}
# @markdown Lưu và dùng lại model tại MyDrive/Colab Notebooks/KoboldCPP; Colab tự tải bằng mạng của runtime.
AllowSaveToGoogleDrive = True # @param {type:"boolean"}
MakeLocalTunnelFallback = False # @param {type:"boolean"}

# --- TỪ ĐIỂN ÁNH XẠ TÊN ĐẸP -> LINK HUGGINGFACE ---
MAP_TEXT_MODELS = {
    "Gemma 4 12B (Q4_K_M)": "https://huggingface.co/unsloth/gemma-4-12b-it-GGUF/resolve/main/gemma-4-12b-it-Q4_K_M.gguf",
    "Gemma 4 E4B Uncensored (Q5_K_M)": "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf",
    "Tiefighter 13B (Q4_K_S)": "https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_S.gguf",
    "Estopia 13B (Q4_K_S)": "https://huggingface.co/KoboldAI/LLaMA2-13B-Estopia-GGUF/resolve/main/LLaMA2-13B-Estopia.Q4_K_S.gguf",
    "Llama 3.1 8B BookAdventures (Q6_K)": "https://huggingface.co/KoboldAI/Llama-3.1-8B-BookAdventures-GGUF/resolve/main/Llama-3.1-8B-BookAdventures.Q6_K.gguf",
    "Cydonia 24B v4.2.0 (Q4_K_S)": "https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v4.2.0-GGUF/resolve/main/TheDrummer_Cydonia-24B-v4.2.0-Q4_K_S.gguf",
    "Broken Tutu 24B (Q4_K_S)": "https://huggingface.co/mradermacher/Broken-Tutu-24B-GGUF/resolve/main/Broken-Tutu-24B.Q4_K_S.gguf",
    "Dans PersonalityEngine 24B (Q4_K_S)": "https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.3.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.3.0-24b-Q4_K_S.gguf",
    "Harbinger 24B (Q4_K_S)": "https://huggingface.co/LatitudeGames/Harbinger-24B-GGUF/resolve/main/Harbinger-24B-Q4_K_S.gguf",
    "Muse 12B (Q4_K_S)": "https://huggingface.co/LatitudeGames/Muse-12B-GGUF/resolve/main/Muse-12B-Q4_K_S.gguf",
    "Qwen3 VL 8B Instruct (Q6_K)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf",
    "Mistral Small 3.2 24B (Q4_K_S)": "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf",
    "GPT OSS 20B (MXFP4)": "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-mxfp4.gguf",
    "Gemma 3 12B (Q4_K_S)": "https://huggingface.co/bartowski/google_gemma-3-12b-it-GGUF/resolve/main/google_gemma-3-12b-it-Q4_K_S.gguf",
    "Gemma 3n E4B (Q6_K)": "https://huggingface.co/unsloth/gemma-3n-E4B-it-GGUF/resolve/main/gemma-3n-E4B-it-Q6_K.gguf",
    "GLM 4 9B (Q6_K)": "https://huggingface.co/unsloth/GLM-4-9B-0414-GGUF/resolve/main/GLM-4-9B-0414-Q6_K.gguf"
}

MAP_MMPROJ = {
    "Gemma 4 E4B mmproj (BF16)": "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf",
    "Qwen3 VL 8B mmproj (BF16)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf",
    "LLaMA 13B mmproj v1.5 (Q4_1)": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf",
    "LLaMA 3 8B mmproj (Q4_1)": "https://huggingface.co/koboldcpp/mmproj/resolve/main/LLaMA3-8B_mmproj-Q4_1.gguf"
}

MAP_IMG_MODELS = {
    "Qwen Image 2.1 (Q8_0 - Max Quality)": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q8_0.gguf",
    "Qwen Image 2.1 (Q5_K_M)": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q5_K_M.gguf",
    "PicX Real (Q8_0)": "https://huggingface.co/koboldcpp/imgmodel/resolve/main/picx_real_q8_0.gguf",
    "DreamShaper XL v2 (Q4_K)": "https://huggingface.co/koboldcpp/imgmodel/resolve/main/dreamshaperXL_v2_q4_k.gguf",
    "Flux Klein 4B (Q4_0)": "https://huggingface.co/leejet/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_0.gguf",
    "Flux Klein 9B (Q5_K_M)": "https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/resolve/main/flux-2-klein-9b-Q5_K_M.gguf"
}

MAP_IMG_CLIPS = {
    "Qwen3 VL 8B Instruct (Q8_0 - Max Text)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q8_0.gguf",
    "Qwen3 VL 8B Instruct (UD-Q4_K_XL - Standard)": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-UD-Q4_K_XL.gguf",
    "Qwen3 4B (Q4_K_M - Low VRAM)": "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf"
}

MAP_IMG_VAES = {
    "Qwen Image 2.1 VAE (Texture-Fix BF16)": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors",
    "Flux2 VAE": "https://huggingface.co/koboldcpp/GGUFDumps/resolve/main/flux2vae.safetensors"
}

MAP_SPEECH_MODELS = {
    "Whisper Base EN (Q5_1)": "https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin"
}

MAP_TTS_MODELS = {
    "Kokoro (Q4)": "https://huggingface.co/koboldcpp/tts/resolve/main/Kokoro_no_espeak_Q4.gguf",
    "OuteTTS 0.2 500M (Q4_0)": "https://huggingface.co/koboldcpp/tts/resolve/main/OuteTTS-0.2-500M-Q4_0.gguf"
}

MAP_WAVTOK_MODELS = {
    "None": "",
    "WavTokenizer Large 75 (Q4_0)": "https://huggingface.co/koboldcpp/tts/resolve/main/WavTokenizer-Large-75-Q4_0.gguf"
}

MAP_EMB_MODELS = {
    "Snowflake Arctic Embed S (Q4_0)": "https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf"
}

# BEGIN EMBEDDED MINIMAX NATIVE
"""Native KoboldCpp MiniMax H3 Image support; embedded into colab.ipynb.

The maintained KoboldCpp SD update (PR #2576) plus the upstream SD source sync
adds H3 Image without replacing KoboldCpp's server or its bundled StableUI.
All weights/builds are downloaded on the Colab runtime. No ComfyUI dependency.
"""
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import tarfile

NATIVE_KCPP_REF = "67399ec7d0c0d7e53d0ee1e940bcebd328b65ed5"
NATIVE_SD_BASE = "73fef08224c50a49961004dc8c6fee2c0f640bcb"
NATIVE_SD_REF = "32d516c0197ed5ca31305b8575e14eb1451fad33"
NATIVE_SOURCE_SHA256 = "3b77a5a277c295a8569c8197dbd95987dab0868bca6c412bfedbf22e0d5da4d0"
NATIVE_PATCH_SHA256 = "dde88ef042473bc5e8a2fdda314f18e371ce05cf15c24c672c4499b2847da230"

MINIMAX_L4_MODELS = {
    "img_model": {
        "repo_id": "AtomicChat/MiniMax-H3-Image-GGUF",
        "revision": "5da8538f0b9a7a8a0afd1ff6ed22ade4b0134094",
        "source_filename": "MiniMax-H3-Image-AD-Q4_K.gguf",
        "size": 11571097824,
        "sha256": "df3f3734c1729765a2b7108bebd69b79d00b85fec5bceaf7363924afa84af1e7",
    },
    "img_clip": {
        "repo_id": "leejet/MiniMax-H3-GGUF",
        "revision": "d9c4c6312b4728a68a15a35626d84775a6523783",
        "source_filename": "qwen3vl_32b_minimax_h3-Q4_K_M.gguf",
        "size": 18218065024,
        "sha256": "11e6efe70a57ce7f4838c47bdbd1a1c4b8ce10e2b7747f1b065990b70f4b05fc",
    },
    "img_vae": {
        "repo_id": "Comfy-Org/MiniMax-H3-Image",
        "revision": "0bb149f51e3156ed4b90dd45d5620346bfe1df8f",
        "source_filename": "vae/minimax_h3_image_vae_fp16.safetensors",
        "size": 1168505312,
        "sha256": "d80e9c2e5cfc9844869961ff5a58c9a9e52182df6bfb30b26018dd7f348721af",
    },
}


def minimax_model_url(model):
    return f"https://huggingface.co/{model['repo_id']}/resolve/{model['revision']}/{model['source_filename']}"


def native_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def native_download(url, target, expected_sha256):
    target = Path(target)
    if target.is_file() and native_file_sha256(target) == expected_sha256:
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + ".part")
    subprocess.run(["curl", "--fail", "--location", "--retry", "3", "--connect-timeout", "30",
                    "--output", str(partial), url], check=True)
    if native_file_sha256(partial) != expected_sha256:
        raise RuntimeError(f"SHA256 mã nguồn không khớp: {partial}")
    partial.replace(target)
    return target


def adapt_native_api_compat(source):
    # Two fork-specific LTX call sites are outside the upstream diff. Match the
    # runner-context signatures introduced by the same SD source sync.
    header = Path(source) / "otherarch/sdcpp/src/model/vae/ltx_audio_vae.hpp"
    data = header.read_bytes()
    for old, new in (
        (b"ggml_ext_pad_ext(ctx->ggml_ctx,\n                                         ctx->backend,",
         b"ggml_ext_pad_ext(ctx,"),
        (b"ggml_ext_slice(ctx->ggml_ctx, latent, 2, 0, config.latent_channels)",
         b"ggml_ext_slice(ctx, latent, 2, 0, config.latent_channels)"),
    ):
        if data.count(old) != 1:
            raise RuntimeError("KoboldCpp thay đổi API LTX ngoài phiên bản đã ghim.")
        data = data.replace(old, new)
    header.write_bytes(data)


def apply_native_sd_patch(source, original_patch):
    # Keep CRLF inside upstream's T5 hunk. read_text()/write_text() breaks it.
    patch = Path(original_patch).read_bytes()
    if hashlib.sha256(patch).hexdigest() != NATIVE_PATCH_SHA256:
        raise RuntimeError("Patch upstream sai SHA256.")
    old = b"     ggml_log_set(sd_ggml_log_callback, nullptr);"
    if patch.count(old) != 1:
        raise RuntimeError("Patch logging upstream thay đổi ngoài dự kiến.")
    # Preserve KoboldCpp's logging hook in the upscaler refactor.
    patch = patch.replace(old, b"     kcpp_sd_ggml_log_set();")
    adapted = Path(source).parent / "sdcpp-kobold.diff"
    adapted.write_bytes(patch)
    command = ["git", "apply", "--directory=otherarch/sdcpp",
               # KoboldCpp uses upstream ggml's CUDA implementation, not SD's custom kernels.
               "--exclude=otherarch/sdcpp/src/core/cuda/*", "--include=otherarch/sdcpp/src/*", str(adapted)]
    subprocess.run(command[:2] + ["--check"] + command[2:], cwd=source, check=True)
    subprocess.run(command, cwd=source, check=True)
    adapt_native_api_compat(source)
    definition = Path(source) / "otherarch/sdcpp/src/model/integrations/minimax_h3.cpp"
    if "VERSION_MINIMAX_H3_IMAGE" not in definition.read_text():
        raise RuntimeError("Backend vừa patch chưa đăng ký H3 Image.")


def minimax_build_info():
    # This preset targets L4 (SM 8.9). CUDA/libc identify compatible cached builds;
    # do not impose RAM/VRAM thresholds or preflight hardware eligibility checks.
    toolkit = subprocess.run(["nvcc", "--version"], check=True, capture_output=True, text=True).stdout
    return {"sm": "89", "cuda": hashlib.sha256(toolkit.encode()).hexdigest()[:12],
            "libc": list(platform.libc_ver()), "source": NATIVE_KCPP_REF, "sd": NATIVE_SD_REF, "recipe": 1}


def configure_minimax_stableui(runtime):
    ui = Path(runtime) / "embd_res/kcpp_sdui.embd"
    data = ui.read_bytes()
    old = b'function oc(){return{steps:20,n:1,sampler_name:"Euler",width:512,height:512,cfg_scale:5,eta:1,flow_shift:0'
    new = b'function oc(){return{steps:6,n:1,sampler_name:"Euler",width:1024,height:1024,cfg_scale:1,eta:1,flow_shift:1'
    if data.count(old) != 1:
        raise RuntimeError("StableUI thay đổi cấu trúc cấu hình; không tự sửa preset.")
    ui.write_bytes(data.replace(old, new))


def ensure_minimax_native(cache_root, *, work_root="/content/koboldcpp-minimax-native"):
    build_info = minimax_build_info()
    identity = hashlib.sha256(json.dumps(build_info, sort_keys=True).encode()).hexdigest()[:16]
    work = Path(work_root) / identity
    runtime = work / "runtime"
    stamp = runtime / "native-build.json"
    if stamp.is_file() and json.loads(stamp.read_text()) == build_info and (runtime / "koboldcpp_cublas.so").is_file():
        return [sys.executable, str(runtime / "koboldcpp.py"), "--skiplauncher"], str(runtime)
    archive = Path(cache_root) / "runtime" / f"koboldcpp-minimax-{identity}.tar.gz"
    receipt = archive.with_suffix(".sha256.json")
    work.mkdir(parents=True, exist_ok=True)
    if archive.is_file() and receipt.is_file():
        record = json.loads(receipt.read_text())
        if record.get("build") != build_info or native_file_sha256(archive) != record.get("sha256"):
            raise RuntimeError(f"Cache build native sai SHA256: {archive}; chuyển file lỗi đi rồi chạy lại.")
        with tarfile.open(archive) as package:
            package.extractall(runtime, filter="data")
        if not stamp.is_file() or json.loads(stamp.read_text()) != build_info or not (runtime / "koboldcpp_cublas.so").is_file():
            raise RuntimeError("Cache native thiếu file; không khởi chạy.")
        print("Dùng lại bản KoboldCpp native đã build trên Drive.", flush=True)
    else:
        if any(shutil.which(tool) is None for tool in ("make", "g++", "git")):
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "build-essential", "git"], check=True)
        source_archive = native_download(
            f"https://codeload.github.com/wbruna/llama.cpp/tar.gz/{NATIVE_KCPP_REF}",
            work / "kobold-source.tar.gz", NATIVE_SOURCE_SHA256)
        patch = native_download(
            f"https://github.com/leejet/stable-diffusion.cpp/compare/{NATIVE_SD_BASE[:7]}...{NATIVE_SD_REF}.diff",
            work / "sdcpp.diff", NATIVE_PATCH_SHA256)
        source = work / ("llama.cpp-" + NATIVE_KCPP_REF)
        patch_stamp = source / ".minimax-patched"
        if not patch_stamp.is_file():
            if source.exists():
                shutil.rmtree(source)
            with tarfile.open(source_archive) as package:
                package.extractall(work, filter="data")
            apply_native_sd_patch(source, patch)
            patch_stamp.write_text(NATIVE_SD_REF)
        print("Build KoboldCpp CUDA + H3 Image cho GPU hiện tại; lần đầu có thể mất nhiều phút.", flush=True)
        # Compile only this GPU architecture. Cache the completed runtime on Drive.
        sm = build_info["sm"]
        nvcc_flags = ("--forward-unknown-to-host-compiler -use_fast_math -extended-lambda "
                      f"-gencode arch=compute_{sm},code=sm_{sm} -DKCPP_LIMIT_CUDA_MAX_ARCH={sm}0")
        subprocess.run(["make", "-j2", "koboldcpp_cublas", "LLAMA_CUBLAS=1", "LLAMA_NO_LLAMAFILE=1",
                        "NVCCFLAGS=" + nvcc_flags], cwd=source, check=True)
        library = source / "koboldcpp_cublas.so"
        if not library.is_file():
            raise RuntimeError("Build chưa tạo koboldcpp_cublas.so.")
        runtime.mkdir(parents=True, exist_ok=True)
        for path in source.glob("*.py"):
            shutil.copy2(path, runtime / path.name)
        shutil.copy2(library, runtime / library.name)
        shutil.copytree(source / "embd_res", runtime / "embd_res", dirs_exist_ok=True)
        configure_minimax_stableui(runtime)
        stamp.write_text(json.dumps(build_info, sort_keys=True))
        archive.parent.mkdir(parents=True, exist_ok=True)
        partial = archive.with_suffix(".gz.part")
        with tarfile.open(partial, "w:gz") as package:
            for path in sorted(runtime.iterdir()):
                package.add(path, arcname=path.name)
        digest = native_file_sha256(partial)
        partial.replace(archive)
        temporary_receipt = receipt.with_suffix(".json.tmp")
        temporary_receipt.write_text(json.dumps({"build": build_info, "sha256": digest}))
        temporary_receipt.replace(receipt)
        print(f"Đã lưu build native lên Drive: {archive}", flush=True)
    return [sys.executable, str(runtime / "koboldcpp.py"), "--skiplauncher"], str(runtime)
# END EMBEDDED MINIMAX NATIVE

PRESETS = {
    "MiniMax H3 Image (Q4 L4)": {
        "native_minimax": True,
        **{key: minimax_model_url(model) for key, model in MINIMAX_L4_MODELS.items()},
    },
    "Gemma4 E4B Uncensored (General)": {
        "model": "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5_K_M.gguf",
        "ctx": "40960",
        "mmproj": "https://huggingface.co/unsloth/gemma-4-E4B-it-GGUF/resolve/main/mmproj-BF16.gguf"
    },
    "Tiefighter 13B (General)": {
        "model": "https://huggingface.co/KoboldAI/LLaMA2-13B-Tiefighter-GGUF/resolve/main/LLaMA2-13B-Tiefighter.Q4_K_S.gguf",
        "ctx": "4096",
        "mmproj": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf"
    },
    "Estopia 13B (General)": {
        "model": "https://huggingface.co/KoboldAI/LLaMA2-13B-Estopia-GGUF/resolve/main/LLaMA2-13B-Estopia.Q4_K_S.gguf",
        "ctx": "4096",
        "mmproj": "https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf"
    },
    "Qwen3 VL 8B (Vision)": {
        "model": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q6_K.gguf",
        "ctx": "32768",
        "mmproj": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/mmproj-BF16.gguf"
    },
    "Cydonia 24B (Roleplay)": {"model": "https://huggingface.co/bartowski/TheDrummer_Cydonia-24B-v4.2.0-GGUF/resolve/main/TheDrummer_Cydonia-24B-v4.2.0-Q4_K_S.gguf", "ctx": "10240"},
    "Broken Tutu 24B (Roleplay)": {"model": "https://huggingface.co/mradermacher/Broken-Tutu-24B-GGUF/resolve/main/Broken-Tutu-24B.Q4_K_S.gguf", "ctx": "10240"},
    "Dans PersonalityEngine 24B (Roleplay)": {"model": "https://huggingface.co/bartowski/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-GGUF/resolve/main/PocketDoc_Dans-PersonalityEngine-V1.2.0-24b-IQ4_XS.gguf", "ctx": "10240"},
    "Harbinger 24B (Instruct)": {"model": "https://huggingface.co/LatitudeGames/Harbinger-24B-GGUF/resolve/main/Harbinger-24B-Q4_K_S.gguf", "ctx": "10240"},
    "Muse 12B (Instruct)": {"model": "https://huggingface.co/LatitudeGames/Muse-12B-GGUF/resolve/main/Muse-12B-Q4_K_S.gguf", "ctx": "49152"},
    "Mistral Small 24B (Corporate)": {"model": "https://huggingface.co/unsloth/Mistral-Small-3.2-24B-Instruct-2506-GGUF/resolve/main/Mistral-Small-3.2-24B-Instruct-2506-Q4_K_S.gguf", "ctx": "10240"},
    "GPT OSS 20B (Corporate)": {"model": "https://huggingface.co/ggml-org/gpt-oss-20b-GGUF/resolve/main/gpt-oss-20b-mxfp4.gguf", "ctx": "32768"},
    "Gemma3 12B (Corporate)": {"model": "https://huggingface.co/bartowski/google_gemma-3-12b-it-GGUF/resolve/main/google_gemma-3-12b-it-Q4_K_S.gguf", "ctx": "16384"},
    "GLM4 9b chat (Corporate)": {"model": "https://huggingface.co/unsloth/GLM-4-9B-0414-GGUF/resolve/main/GLM-4-9B-0414-Q6_K.gguf", "ctx": "49152"},
    "Z-Image Turbo (Image)": {
        "img_model": "https://huggingface.co/leejet/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q4_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-GGUF/resolve/main/Qwen3-4B-Instruct-2507-Q4_K_S.gguf",
        "img_vae": "https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.safetensors"
    },
    "Flux Klein 4B (Image)": {
        "img_model": "https://huggingface.co/leejet/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q4_K_M.gguf",
        "img_vae": "https://huggingface.co/koboldcpp/GGUFDumps/resolve/main/flux2vae.safetensors"
    },
    "Qwen Image 2.1 Max (Q8 L4)": {
        "img_model": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q8_0.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-Q8_0.gguf",
        "img_vae": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors"
    },
    "Qwen Image 2.1 (Image)": {
        "img_model": "https://huggingface.co/unsloth/Qwen-Image-2.1-GGUF/resolve/main/qwen-image-2.1-Q5_K_M.gguf",
        "img_clip": "https://huggingface.co/unsloth/Qwen3-VL-8B-Instruct-GGUF/resolve/main/Qwen3-VL-8B-Instruct-UD-Q4_K_XL.gguf",
        "img_vae": "https://huggingface.co/madebyollin/texture-fix-vae-for-qwen-image-2.1/resolve/main/texture_fix_vae_for_qwen_image_2.1_bf16.safetensors"
    }
}


def configure_models(use_preset, template, manual):
    if not use_preset:
        return dict(manual)
    if template not in PRESETS:
        raise ValueError(f"Preset không tồn tại: {template}")
    print(f"Áp dụng preset: {template}")
    return dict(PRESETS[template])


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


def download_file(url, dest_dir=None, custom_name=None, *, expected_size=None, expected_sha256=None):
    """Runs inside Colab: resume into .part and expose only completed downloads."""
    if not url:
        return None
    parsed = urlparse(url)
    if parsed.scheme != "https" or not parsed.netloc:
        raise ValueError("Model phải là URL HTTPS trực tiếp tới file.")
    filename = custom_name or unquote(Path(parsed.path).name)
    if not filename or Path(filename).name != filename or filename in {".", ".."}:
        raise ValueError("Tên file không hợp lệ.")
    # Distinguish repositories/revisions even when both files are named mmproj.gguf.
    identity = hashlib.sha256(url.encode()).hexdigest()[:16]
    directory = Path(dest_dir or MODEL_CACHE_DIR) / identity
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / filename
    receipt = directory / (filename + ".complete.json")
    partial = directory / (filename + ".part")
    control = directory / (filename + ".part.aria2")

    if target.exists():
        record = json.loads(receipt.read_text()) if receipt.exists() else {}
        known = expected_size is not None and expected_sha256 is not None
        if not known and (record.get("url") != url or not record.get("sha256")):
            raise RuntimeError(f"File chưa có xác nhận tải xong: {target}; kiểm tra hoặc chuyển file đi rồi chạy lại.")
        size = expected_size if known else record["size"]
        digest = expected_sha256 if known else record["sha256"]
        print(f"Kiểm tra cache: {filename} ...", flush=True)
        if target.stat().st_size != size or sha256_file(target) != digest:
            raise RuntimeError(f"Cache sai size/SHA256: {target}; không ghi đè. Chuyển file lỗi đi rồi chạy lại.")
        print(f"Dùng lại model trên {'Drive' if AllowSaveToGoogleDrive else 'runtime'}: {filename}", flush=True)
        return str(target)

    downloaded = partial.stat().st_size if partial.exists() else 0
    if expected_size is not None:
        if downloaded > expected_size:
            raise RuntimeError(f"File tải dở lớn hơn model: {partial}")
        if shutil.disk_usage(directory).free < expected_size - downloaded:
            raise RuntimeError(f"Không đủ dung lượng cho {filename}.")
    print(f"Colab tải → {target} (tiếp tục từ file tải dở nếu có)", flush=True)
    subprocess.run([
        "aria2c", "--console-log-level=warn", "--summary-interval=10",
        "--continue=true", "--auto-file-renaming=false", "--allow-overwrite=true",
        "--file-allocation=none", "--max-tries=5", "--retry-wait=5",
        "--connect-timeout=30", "--timeout=120", "-x", "8", "-s", "8", "-k", "1M",
        "--dir", str(directory), "--out", partial.name, url,
    ], check=True)
    if not partial.is_file() or partial.stat().st_size == 0 or control.exists():
        raise RuntimeError(f"Tải chưa hoàn tất: {partial}. Chạy lại để tiếp tục.")
    if expected_size is not None and partial.stat().st_size != expected_size:
        raise RuntimeError(f"Sai kích thước: {partial}; không sử dụng file này.")
    print(f"Kiểm tra SHA256: {filename} ...", flush=True)
    digest = sha256_file(partial)
    if expected_sha256 is not None and digest != expected_sha256:
        raise RuntimeError(f"Sai SHA256: {partial}; chuyển file lỗi đi trước khi tải lại.")
    record = {"url": url, "size": partial.stat().st_size, "sha256": digest}
    target_temporary_receipt = receipt.with_suffix(".json.tmp")
    target_temporary_receipt.write_text(json.dumps(record), encoding="utf-8")
    partial.rename(target)
    target_temporary_receipt.replace(receipt)
    return str(target)


def download_binary(url, target):
    target = Path(target)
    if not target.is_file() or not os.access(target, os.X_OK):
        partial = target.with_suffix(".part")
        subprocess.run(["curl", "--fail", "--location", "--retry", "3",
                        "--output", str(partial), url], check=True)
        partial.chmod(0o755)
        partial.replace(target)
    return str(target)


def build_kobold_command(config):
    if config.get("native_minimax"):
        prefix, cwd = ensure_minimax_native(CACHE_ROOT)
        command = prefix + ["--usecublas", "--sdoffloadcpu", "--sdflashattention",
                            "--sdclipdevice", "CPU", "--sdvaedevice", "CPU",
                            "--sdvramlimit", "19000", "--sdtiledvae", "512",
                            "--sdclampedsoft", "1024", "--reqtimeout", "3600",
                            "--gendefaults", json.dumps({"steps": 6, "cfg_scale": 1.0,
                                "sampler_name": "euler", "scheduler": "discrete", "width": 1024, "height": 1024})]
    else:
        binary = download_binary("https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64",
                                 "/content/koboldcpp-linux-x64")
        layers = -1 if str(Layers).lower() == "auto" else int(Layers)
        command = [binary, "--gpulayers", str(layers), "--contextsize", str(config.get("ctx", ContextSize)), "--usecublas"]
        cwd = "/content"
    for key, flag, mapping in (
        ("model", "--model", MAP_TEXT_MODELS), ("mmproj", "--mmproj", MAP_MMPROJ),
        ("img_model", "--sdmodel", MAP_IMG_MODELS), ("img_clip", "--sdllm", MAP_IMG_CLIPS),
        ("img_vae", "--sdvae", MAP_IMG_VAES), ("speech", "--whispermodel", MAP_SPEECH_MODELS),
        ("tts", "--ttsmodel", MAP_TTS_MODELS), ("wavtok", "--ttswavtokenizer", MAP_WAVTOK_MODELS),
        ("embeddings", "--embeddingsmodel", MAP_EMB_MODELS),
    ):
        value = config.get(key)
        if value:
            url = mapping.get(value, value)
            if url:
                manifest = MINIMAX_L4_MODELS.get(key) if config.get("native_minimax") else None
                if manifest:
                    path = download_file(url, expected_size=manifest["size"], expected_sha256=manifest["sha256"])
                else:
                    path = download_file(url)
                command.extend([flag, path])
    if config.get("music"):
        music = shlex.split(config["music"])
        for index, arg in enumerate(music[:-1]):
            if arg in {"--musicllm", "--musicembeddings", "--musicdiffusion", "--musicvae"}:
                music[index + 1] = download_file(music[index + 1])
        command.extend(music)
    if Multiplayer:
        command.append("--multiplayer")
    return command, cwd


def run_server(command, cwd):
    process = subprocess.Popen(command, cwd=cwd, start_new_session=True)
    try:
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    finally:
        # Only signal this launcher's process group, including its descendants.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=5)


manual = {
    "model": Model if LoadTextModel else None,
    "mmproj": Mmproj if LoadVisionMMProjector else None,
    "img_model": ImgModel if LoadImgModel else None,
    "img_clip": ImgClip1 if LoadImgModel else None,
    "img_vae": ImgVAE if LoadImgModel else None,
    "speech": SpeechModel if LoadSpeechModel else None,
    "tts": TTSModel if LoadTTSModel else None,
    "wavtok": WavTokModel if LoadTTSModel else None,
    "embeddings": EmbeddingsModel if LoadEmbeddingsModel else None,
    "music": MusicArgs if LoadMusicModel else None,
}
config = configure_models(UsePreset, Template, manual)
LOCAL_CACHE_ROOT = Path("/content/koboldcpp-cache")
if DeleteExistingModels and LOCAL_CACHE_ROOT.exists():
    shutil.rmtree(LOCAL_CACHE_ROOT)
    print("Đã xóa cache tạm của notebook trong /content. Model trên Drive được giữ lại.")
if AllowSaveToGoogleDrive:
    mount = Path("/content/drive")
    if not (mount.is_mount() and (mount / "MyDrive").is_dir()):
        drive.mount(str(mount))
    if not (mount / "MyDrive").is_dir():
        raise RuntimeError("Drive chưa sẵn sàng; hoàn tất cấp quyền Google rồi chạy lại.")
    CACHE_ROOT = mount / "MyDrive/Colab Notebooks/KoboldCPP"
else:
    CACHE_ROOT = LOCAL_CACHE_ROOT
MODEL_CACHE_DIR = CACHE_ROOT / "models"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Cache model: {MODEL_CACHE_DIR}. Tải và đọc model đều do máy Colab thực hiện.")
if shutil.which("aria2c") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], check=True)
cmd_args, working_directory = build_kobold_command(config)

cf_token = ""
try:
    cf_token = (userdata.get("CLOUDFLARE_API_KEY") or "").strip()
except Exception:
    pass

# The custom Cloudflare tunnel must route to http://localhost:5001.
tunnels = []
try:
    if cf_token:
        cloudflared = download_binary("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                                     "/content/cloudflared")
        print("Mở Cloudflare tunnel đã cấu hình tới localhost:5001.")
        tunnels.append(subprocess.Popen([cloudflared, "tunnel", "run", "--token", cf_token],
                                         stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
    else:
        cmd_args.append("--remotetunnel")
    if MakeLocalTunnelFallback:
        subprocess.run(["npm", "install", "-g", "localtunnel"], check=True)
        tunnels.append(subprocess.Popen(["lt", "--port", "5001"]))
        print("Mật khẩu Localtunnel:", flush=True)
        subprocess.run(["curl", "--fail", "--silent", "https://localtunnel.github.io/bypass"], check=True)
    print(f"\nKhởi động KoboldCpp:\n{shlex.join(cmd_args)}", flush=True)
    print("StableUI: thêm /sdui/ vào cuối link tunnel của KoboldCpp.", flush=True)
    run_server(cmd_args, working_directory)
finally:
    cf_token = ""
    for tunnel in tunnels:
        if tunnel.poll() is None:
            tunnel.terminate()
            try:
                tunnel.wait(timeout=5)
            except subprocess.TimeoutExpired:
                tunnel.kill()
                tunnel.wait(timeout=5)
